# Задание 1

In [1]:
import re


def check_car_id(car_id: str) -> str:
    # таблица замены латиница -> кириллица (визуально похожие буквы)
    lat_to_cyr = {
        'A': 'А', 'B': 'В', 'E': 'Е', 'K': 'К',
        'M': 'М', 'H': 'Н', 'O': 'О', 'P': 'Р',
        'C': 'С', 'T': 'Т', 'X': 'Х',
    }

    normalized = ''
    for ch in car_id.upper():
        normalized += lat_to_cyr.get(ch, ch)

    allowed = 'АВЕКМНОРСТУХ'
    pattern = rf'^([{allowed}])(\d{{3}})([{allowed}]{{2}})(\d{{2,3}})$'
    m = re.match(pattern, normalized)

    if m:
        plate = m.group(1) + m.group(2) + m.group(3)
        region = m.group(4)
        return f'Номер {plate} валиден. Регион: {region}.'
    else:
        return 'Номер не валиден.'


print(check_car_id('А222ВС96'))
print(check_car_id('А123ВЕ777'))
print(check_car_id('АБ22ВВ193'))
print(check_car_id('A222BC96'))       # латиница -> нормализуем
print(check_car_id('А222ВС9'))       # регион 1 цифра
print(check_car_id('А222ВС9699'))    # регион 4 цифры
print(check_car_id('Я222ВС96'))      # недопустимая буква
print(check_car_id(''))              # пустая строка

Номер А222ВС валиден. Регион: 96.
Номер А123ВЕ валиден. Регион: 777.
Номер не валиден.
Номер А222ВС валиден. Регион: 96.
Номер не валиден.
Номер не валиден.
Номер не валиден.
Номер не валиден.


# Задание 2

In [2]:
import re


def validate_email(email: str) -> bool:
    pattern = r'^[a-zA-Z0-9][a-zA-Z0-9_.+-]*@[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?(\.[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?)+$'
    return bool(re.match(pattern, email))


def validate_ipv4(ip: str) -> bool:
    pattern = r'^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})$'
    m = re.match(pattern, ip)
    if not m:
        return False
    for octet in m.groups():
        if int(octet) > 255:
            return False
        if len(octet) > 1 and octet.startswith('0'):
            return False
    return True


def validate_card(card_str: str) -> bool:
    """XXXX-XXXX-XXXX-XXXX, ровно 4 группы по 4 цифры"""
    return bool(re.match(r'^\d{4}-\d{4}-\d{4}-\d{4}$', card_str))


def mask_card(card_str: str) -> str:
    last4 = card_str[-4:]
    return f'****-****-****-{last4}'


def remove_consecutive_duplicates(text: str) -> str:
    """Бонус: убирает подряд идущие повторы слов"""
    return re.sub(r'\b(\w+)(\s+\1)+\b', r'\1', text)


def parse_logs(text: str) -> list:
    results = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        # проверяем формат: YYYY-MM-DD HH:MM:SS LEVEL ...
        header = re.match(
            r'^(\d{4}-\d{2}-\d{2})\s(\d{2}:\d{2}:\d{2})\s(INFO|WARN|ERROR|DEBUG)\s(.+)$',
            line
        )
        if not header:
            continue

        ts = f'{header.group(1)} {header.group(2)}'
        level = header.group(3)
        tail = header.group(4)

        user_m = re.search(r'(?:^|\s)user=(\S+)', tail)
        action_m = re.search(r'(?:^|\s)action=(\S+)', tail)
        ip_m = re.search(r'(?:^|\s)ip=(\S+)', tail)
        card_m = re.search(r'(?:^|\s)card=(\S+)', tail)
        msg_m = re.search(r'(?:^|\s)msg=(.*?)(?:\s+\w+=|$)', tail)

        user = user_m.group(1) if user_m else None
        action = action_m.group(1) if action_m else None
        ip = ip_m.group(1) if ip_m else None
        card_raw = card_m.group(1) if card_m else None
        msg = msg_m.group(1) if msg_m else None

        errors = []

        if user is not None and not validate_email(user):
            errors.append('invalid_email')

        if ip is not None and not validate_ipv4(ip):
            errors.append('invalid_ip')

        card = None
        if card_raw is not None:
            if validate_card(card_raw):
                card = mask_card(card_raw)
            else:
                errors.append('invalid_card')

        if msg:
            msg = remove_consecutive_duplicates(msg)

        entry = {
            'ts': ts,
            'level': level,
            'user': user,
            'action': action,
            'ip': ip,
            'card': card,
            'errors': errors,
        }

        if msg:
            entry['msg'] = msg

        results.append(entry)

    return results

In [3]:
# проверка на примере из задания
test_text = """2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5"""

for entry in parse_logs(test_text):
    print(entry)

{'ts': '2024-03-15 14:22:01', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'login', 'ip': '192.168.0.12', 'card': None, 'errors': []}
{'ts': '2024-03-15 14:22:05', 'level': 'WARN', 'user': 'ivan@mail.ru', 'action': 'retry', 'ip': '192.168.0.12', 'card': None, 'errors': []}
{'ts': '2024-03-15 14:23:11', 'level': 'ERROR', 'user': 'bad..email@', 'action': 'pay', 'ip': '999.1.1.1', 'card': '****-****-****-1111', 'errors': ['invalid_email', 'invalid_ip']}


In [4]:
with open('data/data_task4/logs_hw4.txt', 'r', encoding='utf-8') as f:
    log_text = f.read()

parsed = parse_logs(log_text)
print(f'Всего записей: {len(parsed)}')
print()
print('Первые 5:')
for entry in parsed[:5]:
    print(entry)

Всего записей: 586

Первые 5:
{'ts': '2024-03-15 10:51:25', 'level': 'WARN', 'user': 'user+tag@gmail.com', 'action': 'login', 'ip': '203.0.113.42', 'card': None, 'errors': []}
{'ts': '2024-13-01 12:00:00', 'level': 'INFO', 'user': 'ivan@mail.ru', 'action': 'login', 'ip': '1.1.1.1', 'card': None, 'errors': []}
{'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'errors': [], 'msg': 'retry please'}
{'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'errors': ['invalid_ip']}
{'ts': '2024-03-15 09:07:09', 'level': 'ERROR', 'user': 'user@domain.', 'action': 'delete', 'ip': '0.0.0.0', 'card': None, 'errors': ['invalid_email']}


In [5]:
# проверка бонуса - повторы в msg удаляются
count = 0
for entry in parsed:
    if 'msg' in entry:
        print(entry)
        count += 1
        if count >= 4:
            break

{'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'login', 'ip': '8.8.8.8', 'card': None, 'errors': [], 'msg': 'retry please'}
{'ts': '2024-03-15 11:48:05', 'level': 'ERROR', 'user': 'ivan@mail.ru', 'action': 'view', 'ip': '172.16.254.1', 'card': None, 'errors': [], 'msg': 'Напишите функцию которая будет удалять повторы'}
{'ts': '2024-03-15 11:23:26', 'level': 'DEBUG', 'user': 'petr.sidorov@yandex.ru', 'action': 'export', 'ip': '8.8.8.8', 'card': None, 'errors': [], 'msg': 'ok'}
{'ts': '2024-03-15 11:52:20', 'level': 'DEBUG', 'user': 'test_user@sub.domain.org', 'action': 'export', 'ip': '10.0.0.5', 'card': None, 'errors': [], 'msg': 'error timeout'}
